### gender
    female  502
    male    498

### age_group
    21-29  	    200
    30-39	    204
    40-49	    180
    Above 50	416

### ahi_3_remapped
    Prefer not to answer 	39
    bottom         	        339
    middle	                327
    upper	                297

### education_level_remapped
    college_vocational_diploma	229
    other	                    11
    postgraduate	            160
    primary_secondary	        220
    undergraduate	            380

### race_ethnicity
    Asian or Pacific Islander	        231
    Black or African American	        184
    Hispanic or Latino	                166
    Middle Eastern or North African	    32
    White or Caucasian	                313
    Native American or Alaska Native	5
    A race/ethnicity not listed here	69

### extraversion_median
    0         360
    1         640

### agreeableness_median
    0         489
    1         511

### conscientiousness_median
    0         433
    1         567

### neuroticism_median
    0         344
    1         656

### openness_median
    0         474
    1         526

In [1]:
import json
import os
from datetime import datetime
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from PIL import Image
from transformers import AutoProcessor, Qwen3VLForConditionalGeneration


In [2]:
#INITAL CONFIG

PROJECT_DIR = Path.cwd().parent

# options are: "Qwen3-VL-2B" - 
MODEL_NAME =  "Qwen3-VL-2B"

#Create output folders
try:
    os.mkdir((PROJECT_DIR / "outputs" / "inferences").resolve())
except FileExistsError:
    print("Directory already exists.")

#TODO if token not present use NONE
load_dotenv()
hf_token = os.getenv("HF_TOKEN")

Directory already exists.


In [3]:
data = pd.read_csv(f"{PROJECT_DIR}/data/specs_data_safe.csv")
print(data.shape)
data.head(1)

(5000, 52)


,Unnamed: 0,Respondent,Country,gender,age_group,age_group_2,nationality,city_living,city_living_length,ahi,...,left_year,left_path,right_uuid,right_city,right_image_name,right_image_id,right_lat,right_lon,right_year,right_path
0,0,4,Singapore,female,40-49,40+,Singapore,Singapore,Above 5 years,"SGD$82,000 - SGD$99,999",...,2015.0,data/SPECS/svi/Singapore/76d51909-0a1e-4a6b-b0...,2137f518-7738-4109-bab5-ccfc88bd74d6,Santiago,Image_44,44,-33.455094,-70.675773,2016.0,data/SPECS/svi/Santiago/2137f518-7738-4109-bab...


In [4]:

def generate_prompt(row):
    persona_block = f"""Persona profile:
- Country of residence: {row["Country"]}
- Gender: {row["gender"]}
- Age group: {row["age_group"]}
- Education level: {row["education_level_remapped"]}
- Race/ethnicity: {row["race_ethnicity"]}
- Socioeconomic group: {row["ahi_3_remapped"]}

Personality (Big Five, scale: 1 = very low, 5 = very high):
- Extraversion: {row["extraversion"]}
- Agreeableness: {row["agreeableness"]}
- Conscientiousness: {row["conscientiousness"]}
- Neuroticism: {row["neuroticism"]}
- Openness: {row["openness"]}

Adopt the perspective of a person with this profile. Let the profile inform how you interpret the images.
"""

    task_block = """Which place looks safer?

You are shown two street-level images of urban places, Image A and Image B. Their order carries no information.
Judge only from features visible in the images. 

Answer options:
- "image_A": the place in Image A looks safer.
- "image_B": the place in Image B looks safer.
- "equal": the two places look equally safe.

Provide your choice along with 3 specific keywords and a reason that accurately reflect the observed evidence justifying your choice.

Return only a valid JSON object with exactly these keys, no Markdown, no code fences, no other text:

{
    "answer": "<one of: image_A, image_B, equal>",
    "keywords": "<keyword1>, <keyword2>, <keyword3>",
    "justification_image_A": "<visible features only of Image A that affected how safe the place looks>",
    "justification_image_B": "<visible features only of Image B that affected how safe the place looks>"
}
"""

    prompt = persona_block + task_block   # persona condition
    prompt_baseline = task_block          # baseline: same text, persona block removed

    return prompt

def resize_image(image_path, max_size=1024):
    image = Image.open(image_path).convert("RGB")
    image.thumbnail((max_size, max_size), Image.Resampling.LANCZOS)
    return image


In [5]:
# Load Qwen3-VL-2B
if MODEL_NAME == "Qwen3-VL-2B":
    model = Qwen3VLForConditionalGeneration.from_pretrained(
        "Qwen/Qwen3-VL-2B-Instruct",
        token=hf_token,
        dtype="auto",
        device_map="auto"
    )

    processor = AutoProcessor.from_pretrained(
        "Qwen/Qwen3-VL-2B-Instruct"
    )

    model.generation_config.do_sample = False
    model.generation_config.temperature = None
    model.generation_config.top_p = None
    model.generation_config.top_k = None

    print(model.generation_config)

Loading weights:   0%|          | 0/625 [00:00<?, ?it/s]

GenerationConfig {
  "bos_token_id": 151643,
  "do_sample": false,
  "eos_token_id": [
    151645,
    151643
  ],
  "pad_token_id": 151643,
  "repetition_penalty": 1.0
}



In [6]:
answer_mapping = {
    "image_a": "left",
    "image_b": "right",
    "equal": "equal",
}

education_mapping = {
    "college_vocational_diploma": "Vocational college education",
    "other": "Not specified",
    "postgraduate": "Postgraduate education",
    "primary_secondary": "Primary and secondary education",
    "undergraduate": "Undergraduate education",
}

race_mapping = {
    "Asian or Pacific Islander": "Asian or Pacific Islander",
    "Black or African American": "Black or African American",
    "Hispanic or Latino": "Hispanic or Latino",
    "Middle Eastern or  North African": "Middle Eastern or North African",
    "White or Caucasian": "White or Caucasian",
    "Native American or Alaska Native": "Native American or Alaska Native",
    "A race/ethnicity not listed here": "Not specified",
}

data["education_level_remapped"] = (
    data["education_level_remapped"].map(education_mapping).fillna(data["education_level_remapped"])
)
data["race_ethnicity"] = data["race_ethnicity"].map(race_mapping).fillna(data["race_ethnicity"])

In [ ]:
results_df = pd.DataFrame(
    columns=[
        "comparison_id",
        "respondent_id",
        "question",
        "human_answer",
        "model_answer",
        "model_justification_left",
        "model_justification_right",
        "model_keywords",
        "left_image_path",
        "right_image_path",
    ]
)

timestamp = datetime.now().astimezone().strftime("%Y-%m-%d_%H-%M")
now = datetime.now().astimezone().strftime("%H:%M:%S - %d/%m/%Y")
INVERT_IMAGES = False
START = 30
STOP = 60
DOWNSIZE = True  # If true the images will be downsized, useful for testing the expirement pipeline
SIZE = 1024 #768
DO_PERSONA = True  # if False model baseline if True persona profile enabled

print(
    f"Running starting at: {now}\nParameters:\n\t{DO_PERSONA = }\n\t{DOWNSIZE = }\n\t{INVERT_IMAGES = }\n\t{START = } - {STOP = }\n"
)

idx_range = range(START, STOP)


for idx, row in data.iterrows():
    if idx not in idx_range:
        continue

    if row["Question"] == "safe":
        # prompt = generate_prompt(row)
        QUESTION = "Answer the folowing question: Which place looks safer?\n"

        persona_block = f"""Persona profile:
- Country of residence: {row["Country"]}
- Gender: {row["gender"]}
- Age group: {row["age_group"]}
- Education level: {row["education_level_remapped"]}
- Race/ethnicity: {row["race_ethnicity"]}
- Socioeconomic group: {row["ahi_3_remapped"]}

Personality Big Five (scale: 1 = very low, 5 = very high):
- Extraversion: {row["extraversion"]}
- Agreeableness: {row["agreeableness"]}
- Conscientiousness: {row["conscientiousness"]}
- Neuroticism: {row["neuroticism"]}
- Openness: {row["openness"]}

Adopt the perspective of a person with this profile. Let the profile inform how you interpret the images.
"""

        task_block = (
            QUESTION
            + """You are shown two street-level images of urban places, Image A and Image B. Their order carries no information.
Judge only from features visible in the images.
"""
        )

        tail = """
Answer options, pick one:
- "image_A": if the place in Image A looks safer.
- "image_B": if the place in Image B looks safer.
- "equal": if the visible evidence does not support a meaningful difference in perceived safety.

Provide your choice along with 3 specific keywords and a reason that accurately reflect the observed evidence justifying your choice.

Return only a valid JSON object with exactly these keys, no Markdown, no code fences, no other text:

{
    "answer": "<your_answer>",
    "keywords": "<keyword1>, <keyword2>, <keyword3>",
    "justification_image_A": "<visible features only of Image A that affected how safe the place looks>",
    "justification_image_B": "<visible features only of Image B that affected how safe the place looks>"
}
"""
#"justification_image_A": "<visible features only of Image A that affected how safe the place looks>",
 #   "justification_image_B": "<visible features only of Image B that affected how safe the place looks>"
#"justification" : "<Justify your answer>"
        if DO_PERSONA is True:
            head = persona_block + task_block
        else:
            head = task_block
        prompt = head + tail

        if idx == 0:
            print(prompt)

        # Image paths
        left_image_path = (PROJECT_DIR / row["left_path"]).resolve()
        right_image_path = (PROJECT_DIR / row["right_path"]).resolve()

        if DOWNSIZE is False:
            # Load native image
            left_image = Image.open(left_image_path).convert("RGB")
            right_image = Image.open(right_image_path).convert("RGB")
        else:
            # Load resized image
            left_image = resize_image(left_image_path, SIZE)
            right_image = resize_image(right_image_path, SIZE)

        # The order is important:
        # Image A = left image
        # Image B = right image
        if INVERT_IMAGES:  # Used to check if the image order of appearance is influecing the model decision
            left_image, right_image = right_image, left_image
            left_image_path, right_image_path = right_image_path, left_image_path

        messages = [
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": head},
                    {"type": "text", "text": "image_A:"},
                    {"type": "image", "image": left_image},
                    {"type": "text", "text": "image_B:"},
                    {"type": "image", "image": right_image},
                    {"type": "text", "text": tail},
                ],
            }
        ]
        # Prepare text and images together
        inputs = processor.apply_chat_template(
            messages,
            tokenize=True,
            add_generation_prompt=True,
            return_dict=True,
            return_tensors="pt",
        )

        # inputs -> model device
        inputs = inputs.to(model.device)

        generated_ids = model.generate(**inputs, max_new_tokens=500, do_sample=False)

        # removing input tokens
        generated_ids_trimmed = [
            output_ids[len(input_ids) :]
            for input_ids, output_ids in zip(inputs.input_ids, generated_ids)
        ]

        # Decode
        output_text = processor.batch_decode(
            generated_ids_trimmed,
            skip_special_tokens=True,
            clean_up_tokenization_spaces=False,
        )

        print(output_text)

        # Save output
        try:
            result = json.loads(output_text[0])

        except json.JSONDecodeError:
            # If generation ended inside the justification string,
            # close the string and the JSON object
            if not output_text[0].endswith('"'):
                output_text[0] += '"'

            if not output_text.endswith("}"):
                output_text[0] += "}"

            result = json.loads(output_text[0])

        model_answer = answer_mapping.get(result["answer"].strip().lower(), "invalid")

        new_row = {
            "comparison_id": idx,
            "respondent_id": row["Respondent"],
            "question": row["Question"],
            "human_answer": row["Score"],
            "model_answer": model_answer,
            #"model_justification":result["justification"],
            "model_justification_left": result["justification_image_A"],
            "model_justification_right": result["justification_image_B"],
            "model_keywords": result["keywords"].strip().lower(),
            "left_image_path": left_image_path,
            "right_image_path": right_image_path,
        }

        results_df.loc[len(results_df)] = new_row

        results_df.to_csv(
            f"{PROJECT_DIR}/outputs/inferences/mllm_results_{MODEL_NAME}_{timestamp}.csv",
            index=False,
        )
        print(new_row)
        # RTX 2060
        # 37min - 30 inferencias - qwen-2b-vl - imagens reduzidas para 768px - prompt 1

        # 34min - 30 inferencias - qwen-2b-vl - imagens reduzidas para 768px - prompt 2


Running starting at: 21:58:13 - 29/09/2026
Parameters:
	DO_PERSONA = True
	DOWNSIZE = True
	INVERT_IMAGES = False
	START = 30 - STOP = 60

['{\n    "answer": "image_A",\n    "keywords": "pedestrian, street, urban",\n    "justification_image_A": "The street in Image A is bustling with pedestrians and has a high density of people, which suggests a lively and active environment. The presence of a tram and a variety of shops, including a restaurant, indicates a well-developed urban area. The street is paved with bricks and has a clear, organized layout, which contributes to a sense of order and safety. The presence of a large, prominent building in the background adds to the sense of a structured and secure environment.",\n    "justification_image_B": "The street in Image B is less crowded and appears to be more isolated, with fewer people and a more tranquil atmosphere. The area is surrounded by trees and has a more residential feel, with a fence and a clear boundary. The presence of a wa

In [ ]:
print(model.generation_config)